# Estructuras de caso → variantes de rutas clínicas

**Pregunta KISS.** ¿Cómo asignar estructuras de caso clínico candidatas a variantes de rutas clínicas completas para maximizar las asignaciones válidas uno-a-uno antes de generar casos sintéticos?

El producto de esta etapa es un manifiesto de asignación. Todavía no se redactan historias, no se inventan pacientes y no se generan muestras sintéticas.

## Formulación algorítmica

Sea $B=(U,V,E)$ un grafo bipartito:

- $U$: lote piloto **finito** de estructuras de caso, cada una con capacidad unitaria y firma canónica anticlones.
- $V$: variantes clínicas identificadas por `(firma de camino, firma de predicados)`.
- $(u,v)\in E$ si existe una asignación completa $x\in Inst(u)$ tal que $Execute(D,x)=v$.
- $M\subseteq E$: pares estructura–variante sin repetir vértices.

Siguiendo Cormen et al. (4.ª ed., cap. 25), un matching es máximo si no existe un camino aumentante. La implementación contrasta DFS por caminos aumentantes con Hopcroft–Karp, $O(|E|\sqrt{|U|+|V|})$.

## Tres niveles separados

1. **Estructura de caso clínico:** variables, dominios, restricciones y procedencia; aún admite más de una realización.
2. **Instancia:** asignación concreta de valores que fuerza una variante de ruta.
3. **Caso narrado:** realización textual posterior.

Este notebook termina entre los niveles 1 y 2: conserva una asignación testigo por arista, pero no produce el caso final.

In [ ]:
from bipartite_minsa.kiss_case import (
    NO_SEVERITY_ROUTE_VARIANT_ID,
    SEVERITY_PREDICATES,
    build_assignment_manifest,
    build_kiss_case_route_problem,
    execute_kiss_decision_graph,
)
from bipartite_minsa.matching.analysis import hall_deficiency_witnesses
from bipartite_minsa.matching.augmenting_path import AugmentingPathMatcher
from bipartite_minsa.matching.hopcroft_karp import HopcroftKarpMatcher
from bipartite_minsa.matching.oracles import networkx_cardinality

problem = build_kiss_case_route_problem()
graph = problem.graph
print(f'|U|={len(graph.left)}, |V|={len(graph.right)}, |E|={graph.m}')
print('firmas canónicas únicas:', len({u.canonical_signature for u in problem.case_structures}))
assert (len(graph.left), len(graph.right), graph.m) == (7, 8, 14)

## Por qué el grafo no es trivial

Cada estructura contiene los siete criterios observados: uno focal puede ser `true/false` y los otros seis quedan en `false`. Por ello admite dos instancias válidas: una recorre su variante focal con severidad y otra recorre la variante sin severidad. No se aprueba una arista por ausencia de información.

In [ ]:
severity_ids = tuple(item[0] for item in SEVERITY_PREDICATES)
negative_certificates = [
    c for c in problem.certificates
    if c.clinical_route_variant_id == NO_SEVERITY_ROUTE_VARIANT_ID
]
for certificate in negative_certificates:
    witness = certificate.witness
    assert all(witness[predicate_id] is False for predicate_id in severity_ids)
    assert execute_kiss_decision_graph(witness) == NO_SEVERITY_ROUTE_VARIANT_ID
print('aristas a la variante sin severidad con testigo completo:', len(negative_certificates))
print('ejemplo:', negative_certificates[0].witness)

In [ ]:
dfs = AugmentingPathMatcher().solve(graph)
hk = HopcroftKarpMatcher().solve(graph)
nx_cardinality = networkx_cardinality(graph)
print('DFS:', dfs.cardinality)
print('Hopcroft–Karp:', hk.cardinality)
print('NetworkX:', nx_cardinality)
assert dfs.cardinality == hk.cardinality == nx_cardinality == 7
assert len({u for u, _ in hk.pairs}) == len(hk.pairs)
assert len({v for _, v in hk.pairs}) == len(hk.pairs)

## Certificado del límite

El máximo es 7, no 8. El testigo de Hall muestra un conjunto de ocho variantes cuyo vecindario contiene solo siete estructuras: $|S|=8>|N(S)|=7$. El déficit unitario describe capacidad estructural del lote, no calidad clínica.

In [ ]:
witnesses = hall_deficiency_witnesses(graph, hk)
witness = witnesses[0]
print(
    f'Hall: |S|={len(witness.clinical_route_variant_ids)} > '
    f'|N(S)|={len(witness.neighboring_case_structure_ids)}; '
    f'déficit={witness.deficiency}'
)
assert (len(witness.clinical_route_variant_ids), len(witness.neighboring_case_structure_ids)) == (8, 7)

In [ ]:
manifest = build_assignment_manifest(problem, hk)
for item in manifest['selected_assignments']:
    assert item['unit_capacity'] == 1
    assert execute_kiss_decision_graph(item['witness_assignment']) == item['clinical_route_variant_id']
print(manifest['counts'])
print('estado de generación:', manifest['generation_status'])
print('variante sin asignar:', manifest['unassigned_clinical_route_variants'])

## Interpretación y límite

El matching reserva siete estructuras diferentes para siete variantes diferentes y produce un `assignment_manifest.json` reejecutable. No demuestra realismo clínico, seguridad, fidelidad médica de la fuente ni desempeño de un generador. Es la capa combinatoria intermedia entre la GPC estructurada y la futura instanciación sintética.